# Evidence KG Builder × OpenRouter 全链路调试 Notebook（并发版）

这个 Notebook 用来**一步步测试当前 `evidence-kg-builder` Skill**，并直接调用 OpenRouter 模型：

`stealth/space-bunny-alpha`

它不是只看最终答案，而是把整条链路拆开，让你能分别看到：

```text
原始 PDF / DOCX / XLSX / PPTX / 图片
        ↓
Parse：统一成 DocumentUnits
        ↓
Prepare：Python 先做确定性结构抽取与任务分流
        ↓
Extraction：OpenRouter 并发语义抽取
        ↓
Alignment：OpenRouter 并发实体对齐
        ↓
Build：Canonical Facts + Sparse KG
        ↓
Lean GraphRAG：Structured / Fact / Entity Profile / Graph / Raw 多路召回
        ↓
OpenRouter 最终回答
```

### 这个 Notebook 特别适合看什么？

1. **OpenRouter 到底慢在哪里**：每个请求都会记录 wall time、token、失败/重试情况。
2. **Extraction 模型到底抽出了什么**：完整原始模型 JSON 会保存到磁盘，也会在 Notebook 里显示预览。
3. **Alignment 到底合并了什么**：每轮会显示候选数量、模型决策和最终 canonical entity。
4. **Q2 一类召回问题到底卡在哪层**：可以直接查看 `query_plan`、`entity_profiles`、`structured_hits`、`fact_hits`、`graph paths`、`raw units` 和最终 context。
5. **并发是否真的生效**：Extraction 默认 8 路并发、Alignment 默认 8 路并发、批量 QA 默认 4 路并发。

> 这份 Notebook 不把 API Key 写入文件。Key 只通过 `getpass()` 放在当前 Kernel 内存里。

## Cell 1 — 全局配置

**材料切换开关：`DATASET_PROFILE = 'new'`。切回原测试集只需改为 `'original'`，重启内核并运行本格。**

| 配置 | 材料目录 | 构图与日志目录 |
| --- | --- | --- |
| `new`（当前默认） | `materials_new` | `runs/new_material_test_01` |
| `original` | `materials` | `kg_debug_run_q5q10_fix` |

- 将新材料放入 `materials_new`；需要换成外部目录时，只修改 `DATASET_PROFILES['new']['materials']` 的路径。
- 每套配置的 `roles` 单独设置问题集（`questions`）和说明书（`reference`）；其他文件默认为构图语料（`corpus`）。问题集和参考答案不应进入图谱。
- 第一次测试新材料：从 Cell 1 顺序运行到 Cell 18，确认 Cell 17 的 `valid` 为 `true`，再运行问答单元格。
- 切回原材料查询：改为 `'original'`，重启内核并运行 Cell 1，再运行 Cell 20/21、22 或 24；无需重建原图谱。
- 单题 `QUESTION`、批量 `QUESTIONS` 和可选的 `ENTITY_SEARCH` 需要换成对应材料的问题/实体；当前示例仍来自原测试集。
- 问答只搜索当前配置的运行目录，并打印实际图谱路径；新材料未构图时会报错，不会使用原材料回答。
- 原材料找不到完整运行图谱时，可回退到 `examples/prebuilt/work` 的原生示例图谱。
- `PARSER_BACKEND` 可选 `native` 或 `markitdown`。只查询已有图谱不会自动构图。
- 同一套材料续跑时，不要重新运行 Cell 6 的 Fresh Parse，它会重置抽取/对齐状态。


In [8]:
from pathlib import Path

PARSER_BACKEND = 'native'

# 强制禁用旧 skill.zip，直接测试当前 Git 工作区源码
SKILL_ZIP = Path('./__disabled_skill__.zip').resolve()
SKILL_SRC = Path('./skill').resolve()

# 只改这一行切换材料：'new' = 新材料；'original' = 原测试集 A。
DATASET_PROFILE = 'new'

DATASET_PROFILES = {
    'original': {
        'materials': Path('./materials').resolve(),
        'run': Path('./kg_debug_run_q5q10_fix').resolve(),
        'fallback': Path('./examples/prebuilt/work').resolve(),
        'roles': {'测试集A_分析问题集.docx': 'questions'},
    },
    'new': {
        'materials': Path('./materials_new').resolve(),
        'run': Path('./runs/new_material_test_01').resolve(),
        # 新材料未构图时直接报错，不退回原测试集。
        'fallback': None,
        # 如有问题集/任务说明，在这里按相对文件名设置 questions/reference。
        'roles': {},
    },
}
DATASET_CONFIG = DATASET_PROFILES[DATASET_PROFILE]
MATERIALS_ROOT = DATASET_CONFIG['materials']
RUN_ROOT = DATASET_CONFIG['run']
ROLE_OVERRIDES = dict(DATASET_CONFIG['roles'])

SKILL_UNPACK_DIR = RUN_ROOT / 'skill_runtime'
WORK_DIR = RUN_ROOT / ('work' if PARSER_BACKEND == 'native' else 'work_markitdown')
MODEL_OUTPUT_DIR = RUN_ROOT / 'model_outputs'
MANIFEST_PATH = RUN_ROOT / 'inputs.json'

MODEL = 'nvidia/nemotron-3-super-120b-a12b:free'
OPENROUTER_BASE_URL = 'https://openrouter.ai/api/v1'

EXTRACT_REASONING = 'medium'
ALIGN_REASONING = 'medium'
QA_REASONING = 'medium'

EXTRACT_CONCURRENCY = 8
ALIGN_CONCURRENCY = 8
QA_CONCURRENCY = 4

EXTRACT_BATCH_SIZE = 5
EXTRACT_CHAR_BUDGET = 14_000
ALIGN_BATCH_SIZE = 8
# 后续分步 cell 和交互入口都需要这些参数。
EXTRACT_MAX_TOKENS = 10_000
ALIGN_MAX_TOKENS = 3_000
QA_MAX_TOKENS = 16_384
RAW_PREVIEW_CHARS = 5_000
CONTEXT_PREVIEW_CHARS = 8_000
HTTP_TIMEOUT = 300
MAX_NETWORK_RETRIES = 2

# 只搜索当前材料配置的运行目录，防止原材料和新材料的图谱混用。
RAG_SEARCH_ROOT = RUN_ROOT
RAG_FALLBACK_DIR = DATASET_CONFIG['fallback'] or WORK_DIR
RAG_WORK_DIR = None  # 由问答入口在当前配置的范围内自动选择。

if SKILL_ZIP.is_file():
    SKILL_SOURCE, SKILL_SOURCE_KIND = SKILL_ZIP, 'zip'
elif SKILL_SRC.is_dir():
    SKILL_SOURCE, SKILL_SOURCE_KIND = SKILL_SRC, 'dir'
else:
    SKILL_SOURCE, SKILL_SOURCE_KIND = None, 'missing'

RUN_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('SKILL_SOURCE =', SKILL_SOURCE, f'({SKILL_SOURCE_KIND})')
print('材料配置 =', DATASET_PROFILE)
print('材料目录 =', MATERIALS_ROOT)
print('构图 WORK_DIR =', WORK_DIR)
print('图谱搜索目录 =', RAG_SEARCH_ROOT)
print('图谱 fallback =', RAG_FALLBACK_DIR)


SKILL_SOURCE = C:\Users\Agesen\Downloads\Task4\github-release\evidence-kg-graphrag\skill (dir)
材料配置 = new
材料目录 = C:\Users\Agesen\Downloads\Task4\github-release\evidence-kg-graphrag\materials_new
构图 WORK_DIR = C:\Users\Agesen\Downloads\Task4\github-release\evidence-kg-graphrag\runs\new_material_test_01\work
图谱搜索目录 = C:\Users\Agesen\Downloads\Task4\github-release\evidence-kg-graphrag\runs\new_material_test_01
图谱 fallback = C:\Users\Agesen\Downloads\Task4\github-release\evidence-kg-graphrag\runs\new_material_test_01\work


## Cell 2 — 定位 Skill 并安装依赖

这一格做两件事：

1. 定位 Skill 目录：如果 Cell 1 选中了 `skill.zip` 就解压到本次运行目录；如果是已解压的 `./skill/` 就直接用，不做任何复制；
2. 安装 Skill 自己声明的 `requirements.txt`。

运行后会打印：

- Skill 根目录；
- 关键脚本是否存在；
- pip 安装结果。

如果这里失败，后面先不要继续，因为后续所有 Parse / Prepare / Build 都直接调用 Skill 自带脚本。

In [ ]:
import os, sys, shutil, subprocess, json

if 'SKILL_SRC' not in globals() or 'SKILL_SOURCE' not in globals():
    raise RuntimeError('请先运行 Cell 1（全局配置），再运行 Cell 2。')
# Cell 1 的路径也可在内核中手动调整，定位时重新检查实际文件。
SKILL_SOURCE = SKILL_ZIP if SKILL_ZIP.is_file() else SKILL_SRC if SKILL_SRC.is_dir() else None
SKILL_SOURCE_KIND = 'zip' if SKILL_SOURCE == SKILL_ZIP else 'dir' if SKILL_SOURCE is not None else 'missing'
RUN_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert SKILL_SOURCE is not None, (
    f'既找不到 {SKILL_ZIP}，也找不到已解压目录 {SKILL_SRC}'
)

if SKILL_SOURCE_KIND == 'zip':
    if SKILL_UNPACK_DIR.exists():
        shutil.rmtree(SKILL_UNPACK_DIR)
    SKILL_UNPACK_DIR.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(SKILL_SOURCE), str(SKILL_UNPACK_DIR))
    search_root = SKILL_UNPACK_DIR
else:
    # 已经解压好，直接用，不再复制/解压
    search_root = SKILL_SOURCE

def find_skill_dir(root):
    """root 本身或它的单层子目录里，唯一包含 SKILL.md 的那个 Skill 根目录。"""
    if (root / 'SKILL.md').exists():
        return root
    hits = [p for p in root.iterdir() if p.is_dir() and (p / 'SKILL.md').exists()]
    if len(hits) == 1:
        return hits[0]
    raise AssertionError(
        f'在 {root} 里找不到唯一的 Skill 目录（{len(hits)} 个候选：{hits}）'
    )

SKILL_DIR = find_skill_dir(search_root)
SCRIPTS_DIR = SKILL_DIR / 'scripts'
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))
REQUIREMENTS = SKILL_DIR / 'requirements.txt'

print('SKILL_DIR =', SKILL_DIR)
print('scripts:')
for p in sorted(SCRIPTS_DIR.glob('*.py')):
    print('  -', p.name)

if REQUIREMENTS.exists():
    print('\n安装/确认依赖...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REQUIREMENTS)], check=True)
    print('依赖安装完成。')
else:
    print('未发现 requirements.txt，跳过。')
assert PARSER_BACKEND in {'native', 'markitdown'}, '请选择 native 或 markitdown'
if PARSER_BACKEND == 'markitdown':
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(SKILL_DIR / 'requirements-markitdown.txt')], check=True)


## Cell 3 — 查看材料并定义 `corpus / questions / reference`

Skill 的一个重要原则是：**只有 `corpus` 文件能进入知识图谱**。

- `corpus`：真正用于构图的 PDF / Word / Excel / PPT / 图片；
- `questions`：测试题，只能用于提问，不能当事实来源；
- `reference`：任务书、说明文档等，也不能进入 KG。

下面先扫描目录。下一格用 `ROLE_OVERRIDES` 指定非 corpus 文件。

如果你的 `materials` 目录里本来就只放了五份业务语料，那么 `ROLE_OVERRIDES = {}` 即可。

In [11]:
SUPPORTED_SUFFIXES = {'.pdf', '.docx', '.xlsx', '.pptx', '.png', '.jpg', '.jpeg', '.webp', '.md', '.txt'}

assert MATERIALS_ROOT.exists(), f'材料目录不存在: {MATERIALS_ROOT}'
material_files = sorted([
    p for p in MATERIALS_ROOT.rglob('*')
    if p.is_file() and not p.name.startswith('~$') and p.suffix.lower() in SUPPORTED_SUFFIXES
])

print(f'发现 {len(material_files)} 个支持的文件：')
for i, p in enumerate(material_files):
    print(f'[{i:02d}] {p.relative_to(MATERIALS_ROOT)}')

发现 1 个支持的文件：
[00] 英仔爱心社2025年度总结材料(1).docx


## Cell 4 — 生成输入 Manifest

本格使用 Cell 1 当前材料配置的 `ROLE_OVERRIDES`，不会覆盖成原测试集的设置。

如有问题集或任务说明，请在 Cell 1 对应配置的 `roles` 中填写相对路径，例如：

```python
'roles': {
    'questions.docx': 'questions',
    'task.pdf': 'reference',
    'subdir/新问题集.docx': 'questions',
},
```

只有业务语料时保持 `'roles': {}`。运行后检查文件列表和角色统计，确认问题集及参考答案没有被标为 `corpus`。


In [12]:
# 沿用 Cell 1 当前材料配置的角色设置。

entries = []
for p in material_files:
    rel = p.relative_to(MATERIALS_ROOT).as_posix()
    role = ROLE_OVERRIDES.get(rel, 'corpus')
    assert role in {'corpus', 'questions', 'reference'}
    assert '问题集' not in p.stem or role != 'corpus', f'问题集不能标为 corpus: {rel}'
    entries.append({'path': rel, 'role': role})

manifest = {'root': str(MATERIALS_ROOT), 'files': entries}
MANIFEST_PATH.parent.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')

print(json.dumps(manifest, ensure_ascii=False, indent=2))
print('\n角色统计：')
from collections import Counter
print(Counter(x['role'] for x in entries))

{
  "root": "C:\\Users\\Agesen\\Downloads\\Task4\\github-release\\evidence-kg-graphrag\\materials_new",
  "files": [
    {
      "path": "英仔爱心社2025年度总结材料(1).docx",
      "role": "corpus"
    }
  ]
}

角色统计：
Counter({'corpus': 1})


## Cell 5 — CLI 辅助函数

当前 Skill 是一个**文件式协议**：Python 负责 parse / prepare / accept / build；模型只负责 extraction / alignment / final answer。

这个辅助函数统一调用 Skill 的 CLI，并把 stdout 当 JSON 读回来。

如果某一步报错，你会直接看到 Skill 返回的 `error` 和 `error_type`，不会悄悄吞掉。

In [13]:
import subprocess, json, shlex, time
from IPython.display import display, JSON, Markdown

PIPELINE = SCRIPTS_DIR / 'kg_pipeline.py'

def run_cli(*args, check=True):
    cmd = [sys.executable, str(PIPELINE), *map(str, args)]
    proc = subprocess.run(cmd, cwd=str(SKILL_DIR), capture_output=True, text=True, encoding='utf-8')
    stdout = proc.stdout.strip()
    stderr = proc.stderr.strip()
    try:
        data = json.loads(stdout) if stdout else {}
    except Exception:
        data = {'raw_stdout': stdout}
    if check and proc.returncode != 0:
        print('COMMAND:', ' '.join(shlex.quote(x) for x in cmd))
        print('STDOUT:', stdout)
        print('STDERR:', stderr)
        raise RuntimeError(f'CLI failed with code {proc.returncode}')
    return data

## Cell 6 — Fresh Parse：把五种文档统一成 `DocumentUnit`

这里一定使用 `--fresh`，避免复用旧版本的 101 个 Unit 或旧 checkpoint。

### 你应该关注的输出

- `unit_count`：最终切成多少个 DocumentUnit；
- `documents[].status`：每种格式是否成功；
- `cached`：fresh 模式应为 false；
- `parser_versions`：实际使用的解析器版本。

Parse 阶段**不调用 OpenRouter**，所以这里没有模型并发问题。

In [8]:
t0 = time.perf_counter()
if PARSER_BACKEND == 'native':
    parse_result = run_cli('parse', '--manifest', MANIFEST_PATH, '--work', WORK_DIR, '--fresh')
else:
    proc = subprocess.run([sys.executable, str(SCRIPTS_DIR / 'parse_markitdown.py'),
                           '--manifest', str(MANIFEST_PATH), '--work', str(WORK_DIR), '--fresh'],
                          cwd=str(SKILL_DIR), capture_output=True, text=True, encoding='utf-8')
    if proc.returncode != 0:
        print(proc.stdout, proc.stderr)
        raise RuntimeError('MarkItDown parse failed; see the JSON error above')
    parse_result = json.loads(proc.stdout)
parse_elapsed = time.perf_counter() - t0

display(JSON(parse_result))
print(f'\nParse wall time: {parse_elapsed:.2f}s')

<IPython.core.display.JSON object>


Parse wall time: 0.36s


## Cell 7 — 看 `DocumentUnit` 到底长什么样

`00_document_units.json` 是所有后续工作的底座。

每个 Unit 主要包含：

- `format`：pdf/docx/xlsx/pptx/image；
- `locator`：页码、Sheet、行、Slide、Section 等；
- `content_md`：给 LLM 看的紧凑阅读视图；
- `structured`：机器可读的表格/shape/cell ledger；
- `asset_refs`：需要视觉模型查看的图片资产。

这一格会显示格式分布，并抽几个 Unit 给你看。若这里就丢内容，后面再强的模型也救不回来。

In [9]:
import pandas as pd

parsed = json.loads((WORK_DIR / '00_document_units.json').read_text(encoding='utf-8'))
units = parsed['units']
docs = {d['document_id']: d for d in parsed['documents']}

summary_rows = []
for u in units:
    summary_rows.append({
        'unit_id': u['unit_id'],
        'format': u.get('format'),
        'source': docs[u['document_id']]['path'],
        'locator': json.dumps(u.get('locator', {}), ensure_ascii=False),
        'chars': len(u.get('content_md', '')),
        'assets': len(u.get('asset_refs', [])),
        'structured_kind': (u.get('structured') or {}).get('kind'),
    })
unit_df = pd.DataFrame(summary_rows)
display(unit_df.groupby(['format', 'structured_kind'], dropna=False).size().rename('count').reset_index())
display(unit_df.head(12))

print('\n--- 第一个 Unit 的 content_md 预览 ---')
print(units[0].get('content_md', '')[:3000])
print('\nlocator =', units[0].get('locator'))
print('asset_refs =', units[0].get('asset_refs'))

,format,structured_kind,count
0,docx,source_xml_table,6
1,docx,NaN,103


,unit_id,format,source,locator,chars,assets,structured_kind
0,unit_a3af4690866f4c6298f1,docx,英仔爱心社2025年度总结材料(1).docx,"{""section"": [], ""paragraph"": 1}",21,0,NaN
1,unit_c9911a246fcd14403d14,docx,英仔爱心社2025年度总结材料(1).docx,"{""section"": [], ""paragraph_start"": 2, ""paragra...",81,0,NaN
2,unit_5068b2cebd4e6e879f4a,docx,英仔爱心社2025年度总结材料(1).docx,"{""section"": [""社团运营情况""], ""paragraph_start"": 9, ...",8,0,NaN
3,unit_002c3727f13229a6a9b3,docx,英仔爱心社2025年度总结材料(1).docx,"{""section"": [""社团运营情况"", ""社团新媒体平台运营情况""], ""paragr...",14,0,NaN
4,unit_040923713b4cef0c68f8,docx,英仔爱心社2025年度总结材料(1).docx,"{""section"": [""社团运营情况"", ""社团新媒体平台运营情况"", ""新媒体平台基本...",49,0,NaN
5,unit_1ac6806406869aa96209,docx,英仔爱心社2025年度总结材料(1).docx,"{""section"": [""社团运营情况"", ""社团新媒体平台运营情况"", ""新媒体平台基本...",178,0,source_xml_table
6,unit_5549451b9bb0677965de,docx,英仔爱心社2025年度总结材料(1).docx,"{""section"": [""社团运营情况"", ""社团新媒体平台运营情况"", ""新媒体平台建设...",270,0,NaN
7,unit_d7d53df903c4a7a642cb,docx,英仔爱心社2025年度总结材料(1).docx,"{""section"": [""社团运营情况"", ""社团新媒体平台运营情况"", ""新媒体平台建设...",21,0,NaN
8,unit_e85af9152f5fc4788520,docx,英仔爱心社2025年度总结材料(1).docx,"{""section"": [""社团运营情况"", ""社团新媒体平台运营情况"", ""新媒体平台建设...",21,0,NaN
9,unit_5cbd4a1fdbe97ae0eb4e,docx,英仔爱心社2025年度总结材料(1).docx,"{""section"": [""社团运营情况"", ""社团新媒体平台运营情况"", ""新媒体平台建设...",68,0,NaN



--- 第一个 Unit 的 content_md 预览 ---
嵌入图形：需检查原始 DOCX 图形内容。

locator = {'section': [], 'paragraph': 1}
asset_refs = []


## Cell 8 — Prepare：看哪些东西根本不需要 LLM

Prepare 完全是本地 Python：

- 显式 alias table → deterministic；
- 普通 Excel 订单/指标 → deterministic；
- Excel 的备注/原因/风险 → semantic overlay；
- 真正复杂的自然语言 / 图片 → LLM。

### 如何理解结果

- `reviewed_without_llm` 越多，说明 Python 前置处理节省的模型调用越多；
- `pending_llm_units` 是还需要 OpenRouter 的 Unit 数；
- `mixed_semantic_units` 是“结构化主体由 Python 抽，只有自由文本交给模型”的 Excel Unit 数。

In [10]:
t0 = time.perf_counter()
prepare_result = run_cli('prepare', '--work', WORK_DIR)
prepare_elapsed = time.perf_counter() - t0

display(JSON(prepare_result))
print(f'Prepare wall time: {prepare_elapsed:.2f}s')

plan = json.loads((WORK_DIR / '00_extraction_plan.json').read_text(encoding='utf-8'))
plan_df = pd.DataFrame([{
    'unit_id': x['unit_id'],
    'mode': x.get('mode'),
    'priority': x.get('priority'),
    'reason': x.get('reason'),
} for x in plan['units']])
display(plan_df.groupby('mode').size().rename('count').reset_index())
display(plan_df.sort_values(['priority'], ascending=False).head(20))

<IPython.core.display.JSON object>

Prepare wall time: 0.12s


,mode,count
0,auto_no_facts,87
1,llm_batch,18
2,llm_semantic,4


,unit_id,mode,priority,reason
19,unit_fc130afee48cc79f5707,llm_semantic,67,None
31,unit_226ecdce4a368795ae89,llm_semantic,66,None
27,unit_0a16a66d477e89a5aa2f,llm_semantic,66,None
24,unit_8ff2f4a2f2129c632e11,llm_semantic,66,None
21,unit_b77c531eae890fedb0ed,llm_batch,24,None
23,unit_f8ca3b7c2f0bdc011e2e,llm_batch,23,None
29,unit_c2e36ce07d89d90ff602,llm_batch,22,None
6,unit_5549451b9bb0677965de,llm_batch,22,None
13,unit_733099e5eec0b494e76d,llm_batch,21,None
9,unit_5cbd4a1fdbe97ae0eb4e,llm_batch,21,None


## Cell 9 — 输入 OpenRouter API Key，并定义并发模型调用器

API Key 用 `getpass()` 输入，不会显示，也不会写进 Notebook。

### 并发是怎么实现的？

Notebook 使用 `ThreadPoolExecutor`：

```text
Extraction wave
  batch 0 ┐
  batch 1 │
  batch 2 ├── 最多 8 个 HTTP 请求同时发给 OpenRouter
  ...     │
  batch 7 ┘
```

模型请求完成后，**accept 仍然顺序执行**，避免多个线程同时改 `state.json`。

Alignment 也是一样：同一 round 的模型判断并发，接受结果时顺序提交。

每次模型请求都会记录：

- stage / batch；
- task 数；
- wall time；
- prompt / completion / total tokens；
- HTTP / JSON 错误；
- 原始模型文本保存路径。

每次 API 响应会先保存 `model_outputs/*.txt` 和 `*.response.json`，再解析 JSON。失败响应也会保留；`call_logs` 记录 `finish_reason`、输出额度和原始响应路径。接口明确返回 `length` 时，重试自动增加输出额度（最多初始额度的 4 倍）。


响应格式兼容：字符串、`text/output_text` 文本分段、JSON 对象（抽取/对齐）共用 Skill 的 `scripts/model_io.py`。只读取最终文本，不把 reasoning/tool 内容当作结果；空响应有限重试。写每个结果文件前都会确保其父目录存在；本地文件写入失败不会重复请求模型。


In [ ]:
from getpass import getpass
from concurrent.futures import ThreadPoolExecutor, as_completed
import requests, base64, mimetypes, re, traceback

OPENROUTER_API_KEY = getpass('OpenRouter API Key: ').strip()
assert OPENROUTER_API_KEY, 'API Key 不能为空'

call_logs = []

EXTRACTION_SYSTEM = r"""你负责企业异构文档的语义抽取。速度和召回优先。只输出 JSON，不要解释。
对每个 task 只返回 semantic delta：
- mentions: local_id, type, name；只有确有别名/标识时才加 aliases/identifiers。
- relations: subject, predicate, object；可带必要 qualifiers。subject 和 object 必须是单个非空字符串，引用本 task 的 mentions.local_id 或输入 known_mentions.local_id；不得输出列表、字典、null 或数字，不得使用未声明实体的名称。
- 若一个实体与多个实体有关系，拆成多条 relations，每条只引用一个 object；需要的新实体先在 mentions 中声明。不要把多个实体 ID 放入一个 object。
- 文档格式列表、功能列表、数值及其他属性值放入 assertions，不要放入 relations.object；不要为了普通属性值创建实体。
- assertions: subject 必须为单个实体 local_id 字符串；普通属性使用 predicate=attribute、qualifiers.attribute=属性名、value=原始属性值。value 可以是字符串、数字、列表或 JSON 对象；可带必要 qualifiers。
- 例如格式列表应写为 {"subject":"d1","predicate":"attribute","qualifiers":{"attribute":"document_formats"},"value":["PDF","Word"]}，其中 d1 必须已在 mentions 或 known_mentions 中声明。
- 不要重复 known_mentions，不要重述 Python 已经抽出的结构字段。
- 默认不要输出 quote/evidence/review/reason；宿主会自动继承 Unit locator。
- unknown relation predicate 可以保留，不必强行改成白名单。
- unresolved 只写真正影响语义的歧义，不要写格式问题。
可用实体类型优先：Product, AITool, Actor, DataAsset, Version, Event, Observation。
返回一个对象：{"tasks":[...]}。每个输出 task 必须保留输入 task_id。不要输出 Markdown code fence。"""

ALIGNMENT_SYSTEM = r"""你负责实体 cluster 对齐。只做选择，不写分析过程。
对每个 task：比较 source_cluster 与 candidates；若同一稳定实体，decision 返回候选 entity_id；明显新实体返回 NEW_ENTITY；信息确实不足才返回 UNCERTAIN。
优先利用名称/别名/稳定 ID/已确认邻居；不要因为文档上下文不同就保守拆分同一稳定实体。
只能从该 task 给出的 candidate entity_id、NEW_ENTITY、UNCERTAIN 中选择。
返回：{"tasks":[{"task_id":"...","decision":"..."}, ...]}。不要输出 Markdown code fence。"""

def _extract_json_object(content):
    from model_io import model_object
    return model_object(content)


def _normalize_model_text(content):
    from model_io import model_text
    return model_text(content)


def _write_model_text(path, text):
    from model_io import write_model_text
    return write_model_text(path, text)


def _data_url(path: Path):
    mime = mimetypes.guess_type(path.name)[0] or 'image/png'
    data = base64.b64encode(path.read_bytes()).decode('ascii')
    return f'data:{mime};base64,{data}'

def _batch_content(stage, batch, fingerprint, round_no=None):
    envelope = {
        'stage': stage,
        'fingerprint': fingerprint,
        'tasks': batch['tasks'],
    }
    if round_no is not None:
        envelope['round'] = round_no
    text = '处理下面这一批任务，并严格保留每个 task_id：\n' + json.dumps(envelope, ensure_ascii=False, separators=(',', ':'))
    parts = [{'type': 'text', 'text': text}]

    # 视觉任务：把 asset_refs 直接作为多模态图片附加给 OpenRouter。
    seen_assets = set()
    for task in batch['tasks']:
        for ref in task.get('asset_refs', []) or []:
            asset = (WORK_DIR / ref).resolve()
            if not asset.exists() or asset in seen_assets:
                continue
            seen_assets.add(asset)
            parts.append({'type': 'text', 'text': f'下面图片属于 task_id={task["task_id"]}, asset_ref={ref}'})
            parts.append({'type': 'image_url', 'image_url': {'url': _data_url(asset)}})
    return parts

def call_openrouter_batch(stage, batch, fingerprint, round_no=None):
    assert stage in {'extract', 'align'}
    system = EXTRACTION_SYSTEM if stage == 'extract' else ALIGNMENT_SYSTEM
    reasoning = EXTRACT_REASONING if stage == 'extract' else ALIGN_REASONING
    max_tokens = EXTRACT_MAX_TOKENS if stage == 'extract' else ALIGN_MAX_TOKENS
    batch_index = batch.get('batch_index', 0)

    payload = {
        'model': MODEL,
        'messages': [
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': _batch_content(stage, batch, fingerprint, round_no)},
        ],
        'max_tokens': max_tokens,
        'stream': False,
    }
    if reasoning:
        payload['reasoning'] = {'effort': reasoning}

    headers = {
        'Authorization': 'Bearer ' + OPENROUTER_API_KEY,
        'Content-Type': 'application/json',
    }

    last_exc = None
    for attempt in range(MAX_NETWORK_RETRIES + 1):
        t0 = time.perf_counter()
        finish_reason, usage, raw_txt = None, {}, None
        content_type = 'unknown'
        sent_max_tokens = payload['max_tokens']
        try:
            r = requests.post(
                OPENROUTER_BASE_URL.rstrip('/') + '/chat/completions',
                headers=headers,
                json=payload,
                timeout=(30, HTTP_TIMEOUT),
            )
            elapsed = time.perf_counter() - t0
            r.raise_for_status()
            body = r.json()
            choice = (body.get('choices') or [{}])[0]
            content = choice.get('message', {}).get('content', '')
            content_type = type(content).__name__
            finish_reason = choice.get('finish_reason')
            usage = body.get('usage') or {}
            # 先保存响应，再解析：截断或格式错误的响应也能诊断。
            MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
            raw_txt = MODEL_OUTPUT_DIR / f'{stage}_batch_{batch_index:02d}_{time.time_ns()}_attempt{attempt+1}.txt'
            _write_model_text(raw_txt, content if isinstance(content, str) else json.dumps(content, ensure_ascii=False))
            _write_model_text(raw_txt.with_suffix('.response.json'), json.dumps(body, ensure_ascii=False, indent=2))
            if finish_reason == 'length':
                raise ValueError(f'模型输出被截断：finish_reason=length, max_tokens={sent_max_tokens}; 原始响应: {raw_txt}')
            obj = _extract_json_object(content)
            if not isinstance(obj, dict):
                raise ValueError('模型顶层结果不是 JSON object')
            if not isinstance(obj.get('tasks'), list) or not obj['tasks']:
                raise ValueError('模型结果缺少非空 tasks[]')

            # fingerprint / round 是 host 已知元数据，直接补，不浪费模型输出 token。
            obj['fingerprint'] = fingerprint
            if stage == 'align':
                obj['round'] = round_no

            _write_model_text(raw_txt.with_suffix('.json'), json.dumps(obj, ensure_ascii=False, indent=2))

            log = {
                'stage': stage,
                'batch_index': batch_index,
                'tasks': len(batch['tasks']),
                'attempt': attempt + 1,
                'status': 'ok',
                'finish_reason': finish_reason,
                'content_type': content_type,
                'max_tokens': sent_max_tokens,
                'elapsed_s': round(elapsed, 3),
                'prompt_tokens': usage.get('prompt_tokens'),
                'completion_tokens': usage.get('completion_tokens'),
                'total_tokens': usage.get('total_tokens'),
                'cost': usage.get('cost'),
                'raw_path': str(raw_txt),
            }
            call_logs.append(log)
            return {'response': obj, 'raw_content': content, 'body': body, 'log': log}
        except Exception as exc:
            elapsed = time.perf_counter() - t0
            last_exc = exc
            log = {
                'stage': stage, 'batch_index': batch_index, 'tasks': len(batch['tasks']),
                'attempt': attempt + 1, 'status': 'failed', 'elapsed_s': round(elapsed, 3),
                'error': repr(exc), 'finish_reason': finish_reason, 'max_tokens': sent_max_tokens,
                'content_type': content_type,
                'completion_tokens': usage.get('completion_tokens'),
                'total_tokens': usage.get('total_tokens'), 'cost': usage.get('cost'),
                'raw_path': str(raw_txt) if raw_txt else None,
            }
            call_logs.append(log)
            if isinstance(exc, OSError) and not isinstance(exc, requests.RequestException):
                raise  # 本地持久化失败不重复调用模型。
            if attempt >= MAX_NETWORK_RETRIES:
                raise
            # 仅接口明确报告截断时提高额度；不篡改不完整 JSON 或替模型猜 decision。
            if finish_reason == 'length':
                payload['max_tokens'] = min(sent_max_tokens * 2, max_tokens * 4)
                print(f'[{stage}] batch={batch_index} 输出截断，重试 max_tokens={payload["max_tokens"]}')
            time.sleep(1.5 * (2 ** attempt))
    raise last_exc

def run_concurrent_batches(stage, task_export, on_result=None):
    batches = task_export.get('batches') or [
        {'batch_index': 0, 'tasks': task_export.get('tasks', [])}
    ]
    batches = [b for b in batches if b.get('tasks')]
    if not batches:
        return []
    concurrency = EXTRACT_CONCURRENCY if stage == 'extract' else ALIGN_CONCURRENCY
    round_no = task_export.get('round') if stage == 'align' else None
    fingerprint = task_export['fingerprint']

    results, failures = [], []
    with ThreadPoolExecutor(max_workers=min(concurrency, len(batches))) as pool:
        futs = {
            pool.submit(call_openrouter_batch, stage, batch, fingerprint, round_no): batch
            for batch in batches
        }
        for fut in as_completed(futs):
            batch = futs[fut]
            try:
                result = fut.result()
                if on_result is not None:
                    on_result(batch, result)  # 主线程顺序写入检查点。
                results.append((batch, result))
                print(f'[{stage}] batch={batch.get("batch_index")} tasks={len(batch["tasks"])} '
                      f'time={result["log"]["elapsed_s"]:.1f}s tokens={result["log"].get("total_tokens")}')
            except Exception as exc:
                print(f'[{stage}] batch={batch.get("batch_index")} FAILED: {exc!r}')
                failures.append(f'batch={batch.get("batch_index")}: {exc!r}')
    if failures:
        progress = (f'本轮已接收 {len(results)} 个成功批次' if on_result is not None
                    else f'本轮已保存 {len(results)} 个成功响应到 model_outputs')
        raise RuntimeError(progress + '；部分批次失败，可重跑本阶段。\n' + '\n'.join(failures))
    results.sort(key=lambda x: x[0].get('batch_index', 0))
    return results

## Cell 10 — 先生成一波 Extraction 并发任务，但还不调用模型

这一格只调用 Skill 的：

```bash
kg_pipeline.py tasks --stage extract
```

并设置：

- 每个请求最多 5 个 task；
- 每个请求大约最多 14k semantic chars；
- 一次生成最多 8 个互不重叠 batch。

### 你应该看什么？

- `pending_count`：还有多少 Unit 尚未完成语义抽取；
- `batches`：这一波能并发的真实 HTTP 请求；
- 每个 batch 的 task 数 / chars；
- task 中的 `selection.mode`：是自然语言、Excel semantic overlay 还是视觉任务；
- `known_mentions`：Excel deterministic 部分已经抽过的实体，模型不要重复输出。

In [15]:
import pandas as pd

In [13]:
extract_task_file = RUN_ROOT / 'extract_batches_preview.json'
extract_export_meta = run_cli(
    'tasks', '--stage', 'extract', '--work', WORK_DIR,
    '--limit', EXTRACT_BATCH_SIZE,
    '--char-budget', EXTRACT_CHAR_BUDGET,
    '--batch-count', EXTRACT_CONCURRENCY,
    '--output', extract_task_file,
)
extract_export = json.loads(extract_task_file.read_text(encoding='utf-8'))

display(JSON(extract_export_meta))

batch_summary = []
for b in extract_export.get('batches', []):
    batch_summary.append({
        'batch_index': b['batch_index'],
        'tasks': len(b['tasks']),
        'batch_chars': b.get('batch_chars'),
        'modes': ', '.join(t.get('selection', {}).get('mode', '') for t in b['tasks']),
        'assets': sum(len(t.get('asset_refs', [])) for t in b['tasks']),
    })
display(pd.DataFrame(batch_summary))

if extract_export.get('batches'):
    first_task = extract_export['batches'][0]['tasks'][0]
    print('\n--- 第一条 extraction task（结构预览）---')
    display(JSON(first_task))

<IPython.core.display.JSON object>

,batch_index,tasks,batch_chars,modes,assets
0,0,5,13938,"llm_semantic, llm_semantic, llm_semantic, llm_...",0
1,1,5,4737,"llm_semantic, llm_batch, llm_batch, llm_batch,...",0
2,2,5,1308,"llm_batch, llm_batch, llm_batch, llm_batch, ll...",0
3,3,5,684,"llm_batch, llm_batch, llm_batch, llm_batch, ll...",0
4,4,2,277,"llm_batch, llm_batch",0



--- 第一条 extraction task（结构预览）---


<IPython.core.display.JSON object>

## Cell 11 — 真正发起一波 8 路并发 Extraction，并查看模型原始输出

现在才真正访问 OpenRouter。

这一格会：

1. 同时发出最多 8 个 batch；
2. 等模型全部返回；
3. 在 Notebook 中显示每个 batch 的原始文本前 `RAW_PREVIEW_CHARS` 字符；
4. 完整原始输出写到 `kg_debug_run/model_outputs/`；
5. **此时还没有 accept**，方便你先观察模型到底输出了什么。

重点观察：

- 模型有没有仍然重复长 quote/evidence；
- 一个 task 是否只抽真正的 semantic delta；
- Excel semantic overlay 是否只处理备注/风险等自由文本；
- `completion_tokens` 是否明显比旧版小。

In [14]:
extract_wave_results = run_concurrent_batches('extract', extract_export)

for batch, result in extract_wave_results:
    print('\n' + '=' * 100)
    print(f'EXTRACT BATCH {batch["batch_index"]} | {len(batch["tasks"])} tasks | {result["log"]}')
    raw = result['raw_content'] or ''
    print(raw[:RAW_PREVIEW_CHARS])
    if len(raw) > RAW_PREVIEW_CHARS:
        print(f'\n... [Notebook 仅预览前 {RAW_PREVIEW_CHARS} chars；完整结果见 {result["log"]["raw_path"]}]')

[extract] batch=2 tasks=5 time=39.0s tokens=9060
[extract] batch=3 tasks=5 time=40.6s tokens=7745
[extract] batch=4 tasks=2 time=64.9s tokens=4236
[extract] batch=1 tasks=5 time=81.0s tokens=21703
[extract] batch=0 tasks=5 time=72.2s tokens=33215

EXTRACT BATCH 0 | 5 tasks | {'stage': 'extract', 'batch_index': 0, 'tasks': 5, 'attempt': 2, 'status': 'ok', 'elapsed_s': 72.184, 'prompt_tokens': 24991, 'completion_tokens': 8224, 'total_tokens': 33215, 'cost': 0, 'raw_path': 'C:\\Users\\Agesen\\Downloads\\Task4\\github-release\\evidence-kg-graphrag\\runs\\new_material_test_01\\model_outputs\\extract_batch_00_1791371097906.txt'}
{
  "tasks": [
    {
      "task_id": "unit_fc130afee48cc79f5707",
      "mentions": [
        {"local_id":"m1","type":"Actor","name":"英仔爱心社"},
        {"local_id":"m2","type":"Actor","name":"冯琳雅"},
        {"local_id":"m3","type":"Actor","name":"张煜奇"},
        {"local_id":"m4","type":"Actor","name":"尹厚泽"},
        {"local_id":"m5","type":"Actor","name":"吕永奇"},
     

## Cell 12 — 接受这一波 Extraction 结果

模型请求可以并发，但对 `state.json` 的写入不要并发。

所以这里按 batch 顺序执行 `accept`。

当前 speed-first Skill 的 acceptor 很宽松：

- 不要求逐字 quote；
- endpoint 写唯一名称/alias 也会自动 resolve；
- unknown relation predicate 会保留；
- 普通 JSON number 直接接受；
- 一个 item 有问题只跳过那个 item，不会把整 Unit quarantine。

你会看到每次 accept 后还剩多少 `pending_extraction`。

In [ ]:
accept_reports = []
for batch, result in extract_wave_results:
    response_path = MODEL_OUTPUT_DIR / f'accepted_extract_batch_{batch["batch_index"]:02d}_{int(time.time()*1000)}.json'
    _write_model_text(response_path, json.dumps(result['response'], ensure_ascii=False, indent=2))
    report = run_cli('accept', '--stage', 'extract', '--work', WORK_DIR, '--response', response_path)
    accept_reports.append(report)
    print(f'batch {batch["batch_index"]}: accepted={report.get("accepted")} '
          f'pending={report.get("pending_extraction")} reviewed={report.get("reviewed_units")}')

display(pd.DataFrame(accept_reports))

## Cell 13 — 自动跑完剩余 Extraction（仍然 8 路并发）

上一格只跑了一波，是为了让你手工看清模型行为。

这一格会循环：

```text
生成最多 8 个不重叠 batch
        ↓
8 路 OpenRouter 并发
        ↓
顺序 accept
        ↓
直到 pending_extraction = 0
```

每一波仍会把完整原始结果保存到 `model_outputs/`，但 Notebook 只打印摘要，避免页面被几十万字符刷爆。

In [ ]:
def finish_extraction():
    wave = 0
    while True:
        wave += 1
        task_file = RUN_ROOT / f'extract_wave_{wave:02d}.json'
        meta = run_cli(
            'tasks', '--stage', 'extract', '--work', WORK_DIR,
            '--limit', EXTRACT_BATCH_SIZE,
            '--char-budget', EXTRACT_CHAR_BUDGET,
            '--batch-count', EXTRACT_CONCURRENCY,
            '--output', task_file,
        )
        export = json.loads(task_file.read_text(encoding='utf-8'))
        batches = export.get('batches', [])
        if not batches:
            print('Extraction 已全部完成。')
            break
        print(f'\n[Extraction wave {wave}] pending={export.get("pending_count")} batches={len(batches)} '
              f'tasks={sum(len(b["tasks"]) for b in batches)}')
        results = run_concurrent_batches('extract', export)
        for batch, result in results:
            path = MODEL_OUTPUT_DIR / f'extract_wave{wave:02d}_batch{batch["batch_index"]:02d}_accepted.json'
            _write_model_text(path, json.dumps(result['response'], ensure_ascii=False, indent=2))
            rep = run_cli('accept', '--stage', 'extract', '--work', WORK_DIR, '--response', path)
            print(f'  accept batch={batch["batch_index"]} pending={rep.get("pending_extraction")}')

finish_extraction()

## Cell 14 — 检查 Extraction 产物

这里不是做“审查”，而是为了让你肉眼看看模型到底抽成了什么。

重点文件：

- `01_mentions.json`：mention 层实体，尚未全部 canonicalize；
- `state.json`：当前 workflow checkpoint；
- 每个 extraction item 的 `warnings`：只表示 host 做过跳过/修复，不会阻止构图。

你可以看：

- mention 数量和类型分布；
- relation/assertion 总量；
- warning 数量；
- 是否还有大量人工制造的 unresolved（正常情况下应该很少）。

In [17]:
state = json.loads((WORK_DIR / 'state.json').read_text(encoding='utf-8'))
extractions = list(state.get('extractions', {}).values())

mentions = [m for ex in extractions for m in ex.get('mentions', [])]
relations = [r for ex in extractions for r in ex.get('relations', [])]
assertions = [a for ex in extractions for a in ex.get('assertions', [])]
unresolved = [u for ex in extractions for u in ex.get('unresolved', [])]
warnings = [w for ex in extractions for w in ex.get('warnings', [])]

print('extraction units =', len(extractions))
print('mentions         =', len(mentions))
print('relations        =', len(relations))
print('assertions       =', len(assertions))
print('model unresolved =', len(unresolved))
print('host warnings    =', len(warnings))

display(pd.DataFrame(mentions).groupby('type').size().rename('count').reset_index().sort_values('count', ascending=False))
print('\n随机/前几条 relation：')
display(pd.DataFrame(relations).head(15))
print('\n前几条 assertion：')
display(pd.DataFrame(assertions).head(15))

extraction units = 109
mentions         = 130
relations        = 86
assertions       = 60
model unresolved = 0
host warnings    = 4


,type,count
2,Event,65
0,Actor,49
1,DataAsset,16



随机/前几条 relation：


,subject,predicate,object,qualifiers,evidence,origin
0,m_db99ed5d98b51d54dbda,participated_in,m_2d0a6b67020c27e14adf,{},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
1,m_eb15e0f7759b0129783c,led,m_053d656cd15de27d411a,{},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
2,m_9a83f8e4f6760fc44cbb,shared_experience,m_71ef8fee0c8d18479b47,{},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
3,m_523fd5241429306e1426,presented,m_6bebecee387165e4e6e4,{},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
4,m_f1be9c1d64c6b598f316,organized,m_ce0eb57b1bb63b990c50,{},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
5,m_db99ed5d98b51d54dbda,participated_in,m_8370796f05ea5ef43d18,{},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
6,m_eb15e0f7759b0129783c,participated_in,m_1f0cdf646dd58271d3c1,{},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
7,m_9a83f8e4f6760fc44cbb,participated_in,m_556b2934045200f2a617,{},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
8,m_523fd5241429306e1426,participated_in,m_6050eca21b3cd6ccda5f,{},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
9,m_f1be9c1d64c6b598f316,participated_in,m_c81df4da0d4d5e7652bd,{},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted



前几条 assertion：


,subject,predicate,value,qualifiers,evidence,origin
0,m_aba340b9cf7170adac4b,attribute,30,{'attribute': 'activity_count_2024'},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
1,m_aba340b9cf7170adac4b,attribute,True,{'attribute': 'honor_榜样100'},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
2,m_aba340b9cf7170adac4b,attribute,从7087增长至8279,{'attribute': 'public_account_followers_2025_2...,"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
3,m_aba340b9cf7170adac4b,attribute,252,{'attribute': 'bilibili_fans'},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
4,m_aba340b9cf7170adac4b,attribute,37,{'attribute': 'bilibili_videos'},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
5,m_aba340b9cf7170adac4b,attribute,超过600分钟,{'attribute': 'bilibili_total_duration'},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
6,m_aba340b9cf7170adac4b,attribute,777,{'attribute': 'bilibili_likes'},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
7,m_aba340b9cf7170adac4b,attribute,13677,{'attribute': 'bilibili_play_count'},"[{'unit_id': 'unit_fc130afee48cc79f5707', 'doc...",asserted
8,m_da12eefe3d366b87e09c,attribute,30,{'attribute': 'activity_count_2024'},"[{'unit_id': 'unit_226ecdce4a368795ae89', 'doc...",asserted
9,m_da12eefe3d366b87e09c,attribute,True,{'attribute': 'honor_榜样100'},"[{'unit_id': 'unit_226ecdce4a368795ae89', 'doc...",asserted


## Cell 15 — Alignment：按 round 并发实体对齐

Alignment 的并发方式和 Extraction 不同：

```text
Round 1 frozen snapshot
   ├─ batch 0 ┐
   ├─ batch 1 │  8 路并发模型判断
   ├─ ...     │
   └─ batch 7 ┘
          ↓
      顺序 accept
          ↓
如果发生 cluster merge → Round 2
否则停止
```

模型每个 task 只应该返回：

```json
{"task_id":"align_...","decision":"entity_..."}
```

或 `NEW_ENTITY / UNCERTAIN`。

**不需要 reason，不需要 evidence_ids。**

下面会自动跑完所有 round，同时把每轮模型原始输出保存下来。

**失败后续跑：** 保留当前 `WORK_DIR` 和检查点，重跑本格即可。成功批次会在主线程逐个接收并写入检查点，其他批次失败不会抹掉已完成的进度；下一次仅导出尚未完成的任务。不要重跑 Cell 6 的 Fresh Parse。若刚从磁盘重新打开 Notebook，可先运行 Cell 1、2、5、9 初始化环境和模型调用函数，再运行本格。


In [ ]:
def finish_alignment():
    wave = 0
    while True:
        wave += 1
        task_file = RUN_ROOT / f'align_wave_{wave:02d}.json'
        meta = run_cli(
            'tasks', '--stage', 'align', '--work', WORK_DIR,
            '--limit', ALIGN_BATCH_SIZE,
            '--batch-count', ALIGN_CONCURRENCY,
            '--output', task_file,
        )
        export = json.loads(task_file.read_text(encoding='utf-8'))
        batches = export.get('batches', [])
        print(f'\n[Alignment wave {wave}] round={export.get("round")} pending={export.get("pending_count")} '
              f'batches={len(batches)} stopped={export.get("stopped")}')
        if not batches:
            if export.get('stopped'):
                print('Alignment 已停止。')
                break
            # 理论上若没有 task，Skill 会自动 stop；重新取一次状态。
            continue

        # 成功结果立即顺序 accept；单个失败不再阻止同波其他成功批次落盘。
        def accept_alignment(batch, result):
            print(f'  raw decisions batch={batch["batch_index"]}: {str(result["raw_content"])[:1200]}')
            path = MODEL_OUTPUT_DIR / f'align_wave{wave:02d}_batch{batch["batch_index"]:02d}_accepted.json'
            _write_model_text(path, json.dumps(result['response'], ensure_ascii=False, indent=2))
            rep = run_cli('accept', '--stage', 'align', '--work', WORK_DIR, '--response', path)
            print(f'  accept batch={batch["batch_index"]}: round_now={rep.get("round")}')

        run_concurrent_batches('align', export, on_result=accept_alignment)

finish_alignment()

## Cell 16 — 看 Alignment 到底做成什么样

这格主要看 `state.json` 中的：

- `alignment.mapping`：mention → canonical cluster；
- `alignment.clusters`：每个 canonical entity 聚了多少 mention；
- `cluster_status`：resolved / uncertain 等；
- `history`：deterministic merge 和模型 cluster merge 的历史。

重点不是“uncertain 必须为 0”，而是看：

1. 同一稳定产品/工具有没有仍被切成大量碎片；
2. 明显不同版本有没有被错误合并；
3. LLM alignment task 数是否已经比原 mention 数小很多。

In [16]:
state = json.loads((WORK_DIR / 'state.json').read_text(encoding='utf-8'))
al = state['alignment']

print('alignment stopped =', al.get('stopped'))
print('round             =', al.get('round'))
print('clusters          =', len(al.get('clusters', {})))
print('mapping mentions  =', len(al.get('mapping', {})))
print('history entries   =', len(al.get('history', [])))
print('status counts     =', Counter(al.get('cluster_status', {}).values()))

cluster_sizes = sorted(
    [(eid, len(members), al.get('cluster_status', {}).get(eid)) for eid, members in al.get('clusters', {}).items()],
    key=lambda x: -x[1]
)
display(pd.DataFrame(cluster_sizes[:30], columns=['entity_id', 'member_count', 'status']))

print('\n最近 20 条 alignment history：')
display(pd.DataFrame(al.get('history', [])[-20:]))

alignment stopped = True
round             = 3
clusters          = 99
mapping mentions  = 130
history entries   = 52
status counts     = Counter({'record': 65, 'resolved': 33, 'uncertain': 1})


,entity_id,member_count,status
0,entity_6b87174463aa00ff888b,12,resolved
1,entity_11f93804846094735316,5,resolved
2,entity_71c27fc2711c8d7052e7,5,uncertain
3,entity_914c6235c92f2bc44bb8,5,resolved
4,entity_600fea5da0ffa4ca9bfe,5,resolved
5,entity_22f99b06ed49e5f96877,5,resolved
6,entity_5b7e91ee0f16e4d39dbc,1,record
7,entity_031f38b8a989f1dc16d7,1,record
8,entity_7a4c07a8c83aba36f6ab,1,resolved
9,entity_b53b79a5410caf82ea96,1,record



最近 20 条 alignment history：


,round,decision,source_entity_id,target_entity_id,entity_id,source_members,target_members
0,1,cluster_merge,entity_c24ad556ab64d20c41ed,entity_71c27fc2711c8d7052e7,entity_71c27fc2711c8d7052e7,[m_0b6eb2339f7f1c7dae10],"[m_4d697a0585aef65ab6bd, m_f1be9c1d64c6b598f31..."
1,1,cluster_merge,entity_c4f35a9eaf165504c375,entity_22f99b06ed49e5f96877,entity_22f99b06ed49e5f96877,[m_523fd5241429306e1426],"[m_18a75e84b58da991d223, m_7780cf5a49fa10378af..."
2,1,NEW_ENTITY,NaN,NaN,entity_d8cc6f5cee147bbd8219,NaN,NaN
3,1,NEW_ENTITY,NaN,NaN,entity_dc34f01755b334746676,NaN,NaN
4,1,NEW_ENTITY,NaN,NaN,entity_e6d79282f0de5f106009,NaN,NaN
5,1,cluster_merge,entity_e90813f20ed650bbdd46,entity_11f93804846094735316,entity_11f93804846094735316,[m_9a83f8e4f6760fc44cbb],"[m_1349528f942127aa6eb8, m_9e189247e9cfb4c753e..."
6,1,cluster_merge,entity_ed717d3251dab64d5e1a,entity_71c27fc2711c8d7052e7,entity_71c27fc2711c8d7052e7,[m_828ead724d2c548019cd],"[m_0b6eb2339f7f1c7dae10, m_4d697a0585aef65ab6b..."
7,1,NEW_ENTITY,NaN,NaN,entity_f375215ba5a774e3cb53,NaN,NaN
8,1,NEW_ENTITY,NaN,NaN,entity_fdda305703b8837b77dd,NaN,NaN
9,1,NEW_ENTITY,NaN,NaN,entity_5e932475bff52cf0750e,NaN,NaN


## Cell 17 — Build：生成 Canonical Facts 和最终稀疏 KG

Build 阶段不调用 LLM。

它会生成：

- `02_entity_map.json`：mention → canonical entity；
- `03_canonical_facts.json`：事实层 + evidence；
- `04_knowledge_graph.json`：稀疏 traversal graph。

当前 speed-first 版本只做**最小引用完整性检查**，不会再因为 quote、raw literal、domain/range 或 unresolved 数量阻塞 Build。

In [17]:
t0 = time.perf_counter()
build_result = run_cli('build', '--work', WORK_DIR)
build_elapsed = time.perf_counter() - t0

display(JSON(build_result))
print(f'Build wall time: {build_elapsed:.2f}s')

validate_result = run_cli('validate', '--work', WORK_DIR, '--require-complete', check=False)
print('\n--- 最小完整性 validate ---')
display(JSON(validate_result))

<IPython.core.display.JSON object>

Build wall time: 0.16s

--- 最小完整性 validate ---


<IPython.core.display.JSON object>

## Cell 18 — 看最终 KG 的结构和规模

这格用于判断“构图是否太稀/太碎/太肥”。

主要看：

- `nodes`：稳定实体 / Event / Version 等；
- `facts`：绝大多数属性、指标、日期应该留在这里；
- `edges`：真正适合 traversal 的稀疏关系；
- `evidence`：来源 locator；
- `diagnostics`：流程是否完整。

一个健康的 Fact-first KG 往往应该是：**facts 明显多于 nodes/edges**，而不是把每个数字都建成节点。

In [18]:
graph = json.loads((WORK_DIR / '04_knowledge_graph.json').read_text(encoding='utf-8'))
print('nodes    =', len(graph.get('nodes', [])))
print('edges    =', len(graph.get('edges', [])))
print('facts    =', len(graph.get('facts', [])))
print('evidence =', len(graph.get('evidence', [])))
print('unresolved diagnostics =', len(graph.get('diagnostics', {}).get('unresolved', [])))

node_types = Counter(n.get('type') for n in graph.get('nodes', []))
edge_types = Counter(e.get('predicate') for e in graph.get('edges', []))
print('\nNode types:', node_types)
print('\nTop edge predicates:', edge_types.most_common(20))

nodes    = 99
edges    = 86
facts    = 114
evidence = 14
unresolved diagnostics = 0

Node types: Counter({'Event': 65, 'Actor': 18, 'DataAsset': 16})

Top edge predicates: [('participated_in', 40), ('operates', 6), ('shared_experience', 5), ('presented', 5), ('organized', 5), ('led', 5), ('provide', 3), ('属于', 2), ('获得', 2), ('cultivate', 2), ('enhance', 2), ('formulate', 2), ('regulate', 1), ('carry_out', 1), ('construct', 1), ('establish', 1), ('organize', 1), ('awarded_to', 1), ('affiliation', 1)]


## Cell 19 — 可选：按关键词检查某个 canonical entity 的 aliases / alias_bindings

这格对排查跨格式名称题尤其有用。

例如你怀疑“明明 Alignment 已经知道五个别名，为什么回答里没出现”，就把 `ENTITY_SEARCH` 改成其中任意一个名字。

它会直接显示最终 node 的：

- canonical `name`；
- `aliases`；
- `alias_bindings`；
- `attribute_claims`。

如果这里就没有别名，问题在 Extraction/Alignment；如果这里有，但最终回答没有，问题就在 GraphRAG retrieval/context。

In [21]:
ENTITY_SEARCH = '团支书'  # 例如填一个产品名、别名或工具名

if ENTITY_SEARCH.strip():
    q = ENTITY_SEARCH.casefold()
    hits = []
    for n in graph.get('nodes', []):
        labels = [n.get('name', ''), *n.get('aliases', [])]
        if any(q in str(x).casefold() for x in labels):
            hits.append(n)
    print(f'命中 {len(hits)} 个 node')
    for n in hits[:10]:
        display(JSON(n))
else:
    print('ENTITY_SEARCH 为空；需要时填入一个名称后重跑本格。')

命中 0 个 node


## Cell 20 — Lean GraphRAG：只做召回，不调用最终回答模型

这是诊断召回效果最重要的一格。

把 `QUESTION` 改成你要测试的题目。运行后会分别显示：

1. **Query Plan**：Planner 把题目判断成什么类型；
2. **Entity Profiles**：实体名称、aliases、alias_bindings 是否进来了；
3. **Structured Hits**：Sheet/列/单元格/结构化来源；
4. **Fact Hits**：Canonical Fact BM25；
5. **Graph Paths**：只有真正适合多跳图推理的题才应该出现；
6. **Raw Units**：最终回原文补证据的来源；
7. **Context**：真正会送给最终 LLM 的文本。

如果“retrieval 已经找到，但 context 没有”，就是 context budget/组装问题；如果 retrieval 本身没有，就是 Planner 或检索通道问题。
本格可以在新内核中直接运行，无需先执行构图 cell；会加载完整配置和本地依赖，自动读取最新完整图谱，找不到时回退原 native 图谱。


In [ ]:
# 独立准备本 cell 所需环境；不运行安装、解析、抽取、对齐或构图。
from pathlib import Path
from IPython.display import display, JSON, Markdown
import importlib, sys

_query_notebook = Path.cwd() / 'evidence_kg_openrouter_debug.ipynb'
if not _query_notebook.is_file():
    raise FileNotFoundError('请在本 notebook 所在目录启动内核。')
if str(_query_notebook.parent) not in sys.path:
    sys.path.insert(0, str(_query_notebook.parent))
import interactive_kg as _query_module
importlib.reload(_query_module)
_retrieval_runtime = _query_module.query_runtime(_query_notebook, overrides=globals())
call_logs = _retrieval_runtime['call_logs']
RAG_WORK_DIR = _retrieval_runtime['RAG_WORK_DIR']
_selected = _retrieval_runtime['RAG_GRAPH_SELECTION']
print(f'[图谱] source={_selected["source"]} backend={_selected["backend"]} '
      f'nodes={_selected["nodes"]} facts={_selected["facts"]} path={RAG_WORK_DIR}')

import time
import pandas as pd

# 把这里换成你要测试的问题。
QUESTION = """Excel"订单明细"Sheet的备注列中隐藏了6条关键线索（分布在不同的行中）。请找出全部6条隐藏线索，并按时间顺序串联成一条完整的事件链，说明这些事件之间的因果关系。""".strip()
assert QUESTION and '请在这里' not in QUESTION, '请先把 QUESTION 改成真实问题'

LeanGraphRAG = _retrieval_runtime['LeanGraphRAG']
ANSWER_PROMPT = _retrieval_runtime['ANSWER_PROMPT']
rag = LeanGraphRAG(
    RAG_WORK_DIR,
    max_hops=3,
    beam_width=20,
    context_chars=18_000,
)

t0 = time.perf_counter()
retrieval = rag.retrieve(QUESTION)
retrieval_elapsed = time.perf_counter() - t0

print(f'Retrieval wall time: {retrieval_elapsed:.3f}s')
print('\n=== Query Plan ===')
display(JSON(retrieval['query_plan']))

print('\n=== Stats ===')
display(JSON(retrieval['stats']))

print('\n=== Entity Profiles ===')
for x in retrieval.get('entity_profiles', []):
    display(JSON(x))

print('\n=== Structured Hits ===')
structured_rows = [{
    'score': h.get('score'), 'source': h.get('source'), 'locator': str(h.get('locator')),
    'field': h.get('field'), 'value': str(h.get('value'))[:300]
} for h in retrieval.get('structured_hits', [])]
display(pd.DataFrame(structured_rows).head(30))

print('\n=== Fact Hits ===')
fact_rows = []
for h in retrieval.get('fact_hits', []):
    f = h['fact']
    fact_rows.append({
        'score': h.get('score'), 'rrf': h.get('rrf_score'), 'fact_id': h.get('fact_id'),
        'subject': rag.nodes.get(f.get('subject'), {}).get('name'),
        'predicate': f.get('predicate'),
        'object': rag.nodes.get(f.get('object'), {}).get('name') if f.get('object') else None,
        'value': str(f.get('value'))[:220], 'qualifiers': str(f.get('qualifiers'))[:220],
    })
display(pd.DataFrame(fact_rows).head(30))

print('\n=== Graph Paths ===')
for i, p in enumerate(retrieval.get('paths', []), 1):
    chain = []
    for s in p.get('steps', []):
        chain.append(f"{rag.nodes.get(s.get('source'),{}).get('name',s.get('source'))} --{s.get('predicate')}--> {rag.nodes.get(s.get('target'),{}).get('name',s.get('target'))}")
    print(f'Path {i} score={p.get("score"):.3f}: ' + ' | '.join(chain))

print('\n=== Raw Units ===')
raw_rows = [{
    'score': x.get('score'), 'format': x.get('format'), 'source': x.get('source'),
    'locator': str(x.get('locator')), 'preview': x.get('content_md', '')[:500].replace('\n', ' ')
} for x in retrieval.get('raw_units', [])]
display(pd.DataFrame(raw_rows))

print('\n=== 真正送给最终模型的 Context ===')
print('context chars =', len(retrieval['context']))
print(retrieval['context'][:_retrieval_runtime['CONTEXT_PREVIEW_CHARS']])
if len(retrieval['context']) > _retrieval_runtime['CONTEXT_PREVIEW_CHARS']:
    print(f'\n... [仅预览前 {_retrieval_runtime["CONTEXT_PREVIEW_CHARS"]} chars]')

## Cell 21 — 调用 OpenRouter 得到最终答案

这一格使用**刚才已经生成的同一份 GraphRAG context**，直接调用 `stealth/space-bunny-alpha`。

所以如果最终答案有问题，你可以准确区分：

- context 已经有答案但模型没答好 → Final Prompt / 模型问题；
- context 本身缺答案 → Retrieval / 上游 KG 问题。

最终模型被要求直接输出三段 Markdown：

1. `## 1. 结论`
2. `## 2. 推理路径`
3. `## 3. 不确定项`

不走“模型 JSON → Python 再渲染”的额外流程。
先运行 Cell 20，再运行本格；本格自行准备 API 配置，复用已有 Key 或通过密码输入框询问，不需要运行抽取/对齐。


In [ ]:
def call_final_answer(question, retrieval_result):
    from model_io import model_text as _normalize_model_text, write_model_text as _write_model_text
    payload = {
        'model': MODEL,
        'messages': [
            {'role': 'system', 'content': ANSWER_PROMPT},
            {'role': 'user', 'content': retrieval_result['context']},
        ],
        'max_tokens': QA_MAX_TOKENS,
        'stream': False,
    }
    if QA_REASONING:
        payload['reasoning'] = {'effort': QA_REASONING}
    headers = {'Authorization': 'Bearer ' + OPENROUTER_API_KEY, 'Content-Type': 'application/json'}
    last_exc = None
    for attempt in range(MAX_NETWORK_RETRIES + 1):
        t0 = time.perf_counter()
        raw_txt, finish_reason, content_type = None, None, 'unknown'
        try:
            r = requests.post(
                OPENROUTER_BASE_URL.rstrip('/') + '/chat/completions',
                headers=headers, json=payload, timeout=(30, HTTP_TIMEOUT)
            )
            elapsed = time.perf_counter() - t0
            r.raise_for_status()
            body = r.json()
            choice = (body.get('choices') or [{}])[0]
            finish_reason = choice.get('finish_reason')
            content = (choice.get('message') or {}).get('content')
            content_type = type(content).__name__
            usage = body.get('usage') or {}
            raw_txt = MODEL_OUTPUT_DIR / f'qa_{time.time_ns()}_attempt{attempt+1}.txt'
            _write_model_text(raw_txt, content if isinstance(content, str) else json.dumps(content, ensure_ascii=False))
            _write_model_text(raw_txt.with_suffix('.response.json'), json.dumps(body, ensure_ascii=False, indent=2))
            try:
                answer = _normalize_model_text(content)
            except ValueError as exc:
                raise ValueError(f'{exc}（finish_reason={finish_reason!r}; 原始响应: {raw_txt}）') from exc
            if finish_reason == 'length':
                raise ValueError(f'答案被 max_tokens={QA_MAX_TOKENS} 截断'
                                 f'（prompt_tokens={usage.get("prompt_tokens")}）')
            call_logs.append({
                'stage': 'qa', 'batch_index': 0, 'tasks': 1, 'attempt': attempt + 1, 'status': 'ok',
                'elapsed_s': round(elapsed, 3), 'finish_reason': finish_reason,
                'content_type': content_type, 'raw_path': str(raw_txt),
                'prompt_tokens': usage.get('prompt_tokens'), 'completion_tokens': usage.get('completion_tokens'),
                'total_tokens': usage.get('total_tokens'), 'cost': usage.get('cost'),
            })
            return answer, usage, elapsed
        except Exception as exc:
            last_exc = exc
            if isinstance(exc, OSError) and not isinstance(exc, requests.RequestException):
                raise
            if attempt >= MAX_NETWORK_RETRIES:
                call_logs.append({
                    'stage': 'qa', 'batch_index': 0, 'tasks': 1, 'attempt': attempt + 1,
                    'status': 'failed', 'elapsed_s': round(time.perf_counter() - t0, 3),
                    'error': repr(exc), 'finish_reason': finish_reason, 'content_type': content_type,
                    'raw_path': str(raw_txt) if raw_txt else None,
                })
                raise
            wait_s = 1.5 * (2 ** attempt)
            print(f'[qa] 第 {attempt + 1} 次失败：{exc!r}；{wait_s:.1f}s 后重试')
            time.sleep(wait_s)
    raise last_exc

if 'QUESTION' not in globals() or 'retrieval' not in globals():
    raise RuntimeError('请先运行 Cell 20，得到同一问题的检索上下文，再运行 Cell 21。')

# 独立准备本 cell 所需环境；不运行安装、解析、抽取、对齐或构图。
from pathlib import Path
from IPython.display import display, Markdown
import importlib, sys

_query_notebook = Path.cwd() / 'evidence_kg_openrouter_debug.ipynb'
if not _query_notebook.is_file():
    raise FileNotFoundError('请在本 notebook 所在目录启动内核。')
if str(_query_notebook.parent) not in sys.path:
    sys.path.insert(0, str(_query_notebook.parent))
import interactive_kg as _query_module
importlib.reload(_query_module)
_single_runtime = _query_module.query_runtime(_query_notebook, overrides=globals())
call_logs = _single_runtime['call_logs']
RAG_WORK_DIR = _single_runtime['RAG_WORK_DIR']
if 'rag' in globals() and Path(rag.work).resolve() != RAG_WORK_DIR:
    raise RuntimeError('问答图谱已变化，请重新运行 Cell 20 获取新图谱的检索上下文，再运行 Cell 21。')
if not _single_runtime['OPENROUTER_API_KEY']:
    from getpass import getpass
    _single_runtime['OPENROUTER_API_KEY'] = getpass('OpenRouter API Key: ').strip()
if not _single_runtime['OPENROUTER_API_KEY']:
    raise ValueError('API Key 不能为空。')

answer, qa_usage, qa_elapsed = _single_runtime['call_final_answer'](QUESTION, retrieval)
print(f'QA wall time: {qa_elapsed:.2f}s')
print('usage =', qa_usage)
print('\n' + '=' * 100 + '\n')
display(Markdown(answer))

## Cell 22 — 可选：一次并发测试多道题

如果你已经完成构图，后面的每道 QA 彼此独立，可以 4 路并发。

把问题放进 `QUESTIONS`。每道题会先在本地完成 Lean GraphRAG retrieval，再并发调用 OpenRouter 最终回答。

这格特别适合你一次快速测 Q1/Q2/Q5 等不同题型，看：

- cross-format identity 是否走 Entity Profile / source-aware retrieval；
- Excel Sheet/列扫描是否走 Structured；
- 真正多跳因果题是否才走 weighted beam。
本格可直接运行；保留当前 `QUESTIONS` 列表。无需运行 Cell 9 或构图部分，调用日志与 Cell 23、Cell 24 共享。


In [ ]:
# 独立准备本 cell 所需环境；不运行安装、解析、抽取、对齐或构图。
from pathlib import Path
from IPython.display import display, Markdown
import importlib, sys

_query_notebook = Path.cwd() / 'evidence_kg_openrouter_debug.ipynb'
if not _query_notebook.is_file():
    raise FileNotFoundError('请在本 notebook 所在目录启动内核。')
if str(_query_notebook.parent) not in sys.path:
    sys.path.insert(0, str(_query_notebook.parent))
import interactive_kg as _query_module
importlib.reload(_query_module)
_qa_runtime = _query_module.query_runtime(_query_notebook, overrides=globals())
call_logs = _qa_runtime['call_logs']
RAG_WORK_DIR = _qa_runtime['RAG_WORK_DIR']
if not _qa_runtime['OPENROUTER_API_KEY']:
    from getpass import getpass
    _qa_runtime['OPENROUTER_API_KEY'] = getpass('OpenRouter API Key: ').strip()
if not _qa_runtime['OPENROUTER_API_KEY']:
    raise ValueError('API Key 不能为空。')

from concurrent.futures import ThreadPoolExecutor, as_completed

_selected = _qa_runtime['RAG_GRAPH_SELECTION']
print(f'[图谱] source={_selected["source"]} backend={_selected["backend"]} '
      f'nodes={_selected["nodes"]} facts={_selected["facts"]} path={RAG_WORK_DIR}')

QUESTIONS = [
    "2026年第二季度，明创工坊的明星产品\"AI代码助手插件\"季度收入从12.8万元降至9.1万元，下降29%。请按时间顺序列出Q2期间与该产品收入下降相关的全部变更事件，并说明每个事件与收入下降的因果关系。注意：相关线索分散在Excel（含隐藏的备注列）、Word、PDF、PPT等多种文档中，且该产品在不同文档中使用不同名称。",
    "\"AI代码助手插件\"这一产品在PDF、Word、Excel、PPT、图片5种格式文档中分别使用了什么名称？每种格式中包含哪些仅在该格式中可见的独有信息？请逐一列出。",
    "客户\"程序员老王\"购买了\"AI代码助手插件\"的年度订阅。请给出从该客户到产品收入下降的完整关联链路，包括：客户→产品→依赖的AI工具→定价策略变更→续费结果。标注每跳的信息来源。",
    "PDF文档中的产品定价规格表（T1）在跨页续表中省略了\"定价\"和\"适用平台\"两列。对于续表中出现的产品，请说明如何通过其他文档（Word/Excel/PPT）还原这些缺失的定价和平台信息，并给出至少2个具体产品的还原路径。",
    "Excel\"订单明细\"Sheet的备注列中隐藏了6条关键线索（分布在不同的行中）。请找出全部6条隐藏线索，并按时间顺序串联成一条完整的事件链，说明这些事件之间的因果关系。",
    "\"设计素材包V3\"被\"设计素材包V4\"替代。请说明：(1)替代的原因是什么？(2)V4覆盖了V3的哪些内容？(3)V4上线后的用户评分反馈如何？(4)这个替代决策记录在哪些文档中？",
    "2026年4月8日的定价策略变更（免费增值→订阅制）对哪些产品和客户影响最大？请列出受影响最大的3个产品及其影响数据，并说明哪些产品不受影响及原因。",
    "Cursor v3即将发布，\"AI代码助手插件\"面临什么风险？请根据Word文档中的风险预警说明，分析可能的影响路径和张明的应对建议。",
    "编号\"RV-2026-0412\"在PPT中出现在两个不同的场景中。请说明这两个场景分别是什么，该编号被重复使用是否会造成歧义？在构建知识图谱时如何处理这种编号重复的情况？",
    "PDF文档中的AI工具使用对照表（T2）的续表清空。请根据首页表头和续表数据，还原正确的列对应关系，并说明判断依据。同时，该表中哪些AI工具与\"AI代码助手插件\"存在直接或间接的关联？",
]

def answer_one_question(index, q):
    local_rag = _qa_runtime['LeanGraphRAG'](RAG_WORK_DIR, max_hops=3, beam_width=20, context_chars=18_000)
    ret = local_rag.retrieve(q)
    ans, usage, elapsed = _qa_runtime['call_final_answer'](q, ret)
    return {
        'index': index, 'question': q, 'answer': ans, 'usage': usage,
        'elapsed_s': elapsed, 'query_plan': ret['query_plan'], 'stats': ret['stats']
    }

def run_concurrent_questions(questions, concurrency):
    results = []
    with ThreadPoolExecutor(max_workers=min(concurrency, len(questions))) as pool:
        futs = {pool.submit(answer_one_question, i, q): i for i, q in enumerate(questions, 1)}
        for fut in as_completed(futs):
            index = futs[fut]
            try:
                res = fut.result()
            except Exception as exc:
                # 单题失败不应打断整批并发；记录后继续，让失败可见而不是空白。
                print(f'Q{index} FAILED: {exc!r}')
                res = {'index': index, 'question': questions[index - 1], 'answer': '',
                       'usage': {}, 'elapsed_s': 0.0, 'query_plan': {'intent': 'error'},
                       'stats': {}, 'error': repr(exc)}
            results.append(res)
            stats = res['stats']
            if not stats:
                continue
            print(f'Q{res["index"]} finished in {res["elapsed_s"]:.1f}s '
                  f'| plan={res["query_plan"]["intent"]} '
                  f'mode={stats["retrieval_mode"]} '
                  f'structured={stats["structured"]} '
                  f'scope={stats["structured_scope_matches"]} '
                  f'augmentation={stats["structured_selected_for_augmentation"]} '
                  f'table_units={stats.get("table_units", 0)} '
                  f'table_columns={stats.get("table_columns", 0)} '
                  f'table_rows={stats.get("table_context_rows", 0)}/{stats.get("table_rows", 0)} '
                  f'timeline={stats.get("timeline_context_events", 0)}/{stats.get("timeline_events", 0)} '
                  f'paths={stats["paths"]}')
    results.sort(key=lambda x: x['index'])
    return results

if QUESTIONS:
    multi_results = run_concurrent_questions(QUESTIONS, _qa_runtime['QA_CONCURRENCY'])
    for res in multi_results:
        print('\n' + '=' * 110)
        print(f'Q{res["index"]}: {res["question"]}')
        if res.get('error'):
            print(f'[FAILED] {res["error"]}')
            continue
        display(Markdown(res['answer']))
else:
    print('QUESTIONS 为空；需要批量测试时填入问题列表后重跑本格。')

## Cell 23 — 汇总整次 OpenRouter 性能

最后把所有模型调用汇总成表。

最值得看的指标：

- `elapsed_s`：单请求墙钟时间；
- `prompt_tokens`：Alignment 如果仍然很高，说明 candidate profile 还太肥；
- `completion_tokens`：Extraction 如果仍然动辄上万，说明模型还在重复 evidence / 原文；
- `stage` 聚合后的总 token 和平均耗时；
- failed request / retry 情况。

**并发情况下不要把每个请求 elapsed 直接相加当总墙钟。**它反映的是 provider 单请求延迟；真正总耗时应该看每个阶段从开始到结束的 wall time。
本格可在 Cell 21、22 或 24 后运行；没有请求时显示空日志，不会调用模型或构图。


In [ ]:
# 独立准备本 cell 所需环境；不运行安装、解析、抽取、对齐或构图。
from pathlib import Path
from IPython.display import display, Markdown
import importlib, sys

_query_notebook = Path.cwd() / 'evidence_kg_openrouter_debug.ipynb'
if not _query_notebook.is_file():
    raise FileNotFoundError('请在本 notebook 所在目录启动内核。')
if str(_query_notebook.parent) not in sys.path:
    sys.path.insert(0, str(_query_notebook.parent))
import interactive_kg as _query_module
importlib.reload(_query_module)
_metrics_runtime = _query_module.query_runtime(_query_notebook, overrides=globals())
call_logs = _metrics_runtime['call_logs']
RAG_WORK_DIR = _metrics_runtime['RAG_WORK_DIR']

import pandas as pd
import json

log_df = pd.DataFrame(call_logs)
if len(log_df):
    display(log_df)
    print('\n=== 按 stage 聚合 ===')
    agg = log_df.groupby('stage').agg(
        requests=('stage', 'size'),
        avg_latency_s=('elapsed_s', 'mean'),
        p95_latency_s=('elapsed_s', lambda s: s.quantile(0.95)),
        prompt_tokens=('prompt_tokens', 'sum'),
        completion_tokens=('completion_tokens', 'sum'),
        total_tokens=('total_tokens', 'sum'),
    ).reset_index()
    display(agg)

    metrics_path = _metrics_runtime['RUN_ROOT'] / 'openrouter_metrics.json'
    metrics_path.parent.mkdir(parents=True, exist_ok=True)
    metrics_path.write_text(json.dumps(call_logs, ensure_ascii=False, indent=2), encoding='utf-8')
    print('完整调用日志已保存：', metrics_path)
else:
    print('当前还没有 OpenRouter 调用日志。')

# 结果应该怎么判断？

跑完以后，可以用下面这套非常直接的判断标准：

### Parse / Prepare 健康

- 所有 corpus 文档 `status=success`；
- DocumentUnit 中能看到原始关键内容和 locator；
- Excel 普通结构走 deterministic，自由文本走 semantic overlay；
- 不应该因为某个 Unit 没“完美 evidence”而停流程。

### Extraction 健康

- 模型输出主要是短 JSON delta；
- completion token 明显低于旧版那种每批一两万甚至两万多 token 的水平；
- `model unresolved` 只是真歧义，不是格式审计垃圾；
- unknown predicate / 普通数值不会把整 task 弄失败。

### Alignment 健康

- 明显同一稳定实体能合；
- 不同版本/强 ID 冲突不会误合；
- 每个 task 只输出 decision；
- cluster 数明显小于 raw mention 数；
- `UNCERTAIN` 不需要追求 0，只要别因为“上下文不同”过度拆分。

### GraphRAG 健康

- Sheet/列问题：`structured_hits` 应该是主力；
- 跨格式名称题：`query_plan.intent` 应是 `cross_format_identity`，`entity_profiles` 应包含 aliases / alias_bindings，`paths` 通常应为 0；
- 多跳因果题：才应该出现 weighted beam graph paths；
- `raw_units` 应覆盖题目明确要求的来源格式；
- 最重要：**最终 context 里必须真的包含你希望模型回答的证据。**

### Final Answer 健康

必须直接出现：

```text
## 1. 结论
## 2. 推理路径
## 3. 不确定项
```

只有会改变主要结论的关键缺失/冲突才进入“不确定项”，不要再输出大量防御性免责声明。

## Cell 24 — 交互问答：自动选择已有 GraphRAG

运行本格，填写 API Key、输入问题并点击“提问”。

运行时自动寻找当前 `PARSER_BACKEND` 最新的完整图谱，找不到时回退之前的 native 图谱。上方 `[图谱]` 日志显示实际选择的路径、后端、latest/fallback 和规模。

此入口只查询已有图谱，不会自动解析、抽取、对齐或构图。新图谱生成后，重新运行本格即可自动切换到它。


In [ ]:
from pathlib import Path
import importlib, sys

_chat_notebook = Path.cwd() / 'evidence_kg_openrouter_debug.ipynb'
if not _chat_notebook.is_file():
    raise FileNotFoundError('请在本 notebook 所在目录启动内核，或将 _chat_notebook 改为它的完整路径。')
if str(_chat_notebook.parent) not in sys.path:
    sys.path.insert(0, str(_chat_notebook.parent))
import interactive_kg as _interactive_kg
importlib.reload(_interactive_kg)

# 与单题/并发入口使用同一套自动选择规则；不触发构图。
_chat_runtime = _interactive_kg.query_runtime(_chat_notebook, overrides=globals())
RAG_WORK_DIR = _chat_runtime['RAG_WORK_DIR']
call_logs = _chat_runtime['call_logs']
_selected = _chat_runtime['RAG_GRAPH_SELECTION']
print(f'[图谱] source={_selected["source"]} backend={_selected["backend"]} '
      f'nodes={_selected["nodes"]} facts={_selected["facts"]} path={RAG_WORK_DIR}')
KG_CHAT = _interactive_kg.InteractiveKGSession(
    _chat_notebook, overrides=_chat_runtime, graph_work_dir=RAG_WORK_DIR,
)
KG_CHAT_UI = _interactive_kg.display_chat(KG_CHAT)


[图谱] source=fallback backend=native nodes=99 facts=114 path=C:\Users\Agesen\Downloads\Task4\github-release\evidence-kg-graphrag\runs\new_material_test_01\work


In [ ]:
from lean_graphrag import LeanGraphRAG

rag = LeanGraphRAG(
    r"C:\Users\Agesen\Downloads\Task4\github-release\evidence-kg-graphrag\examples\prebuilt\work",
    context_chars=18000,
)

q = 'Excel"订单明细"Sheet的备注列中隐藏了6条关键线索（分布在不同的行中）。请找出全部6条隐藏线索，并按时间顺序串联成一条完整的事件链，说明这些事件之间的因果关系。'

ret = rag.retrieve(q)

print("=== QUERY PLAN ===")
print(ret["query_plan"])

print("\n=== STRUCTURED HITS ===")
print("count =", len(ret["structured_hits"]))

for hit in ret["structured_hits"]:
    print(
        hit.get("locator"),
        "| field =", hit.get("field"),
        "| value =", hit.get("value")
    )

print("\n=== AUGMENTATION QUERY ===")
print(ret.get("augmentation_query"))